# GoogleColabで学ぶデータ分析——クレジットカード不正利用検知

このノートブックでは、約28万件のクレジットカード取引データを使い、
**ランダムフォレスト・XGBoost・LightGBM**の3つのモデルを実際に動かして比較します。

セルは上から順番に実行してください。各セルの上に説明を書いていますので、コードの意味を確認しながら進めてみてください。

## データの出典・ライセンスについて

本記事で使用するデータは、Worldline社とブリュッセル自由大学（ULB）Machine Learning Groupが作成し、
Kaggle上で公開している「Credit Card Fraud Detection」データセットです。

- 公式ページ：https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
- ライセンス：Database Contents License (DbCL) v1.0（出典を明記すれば商用利用を含め自由に利用可能）
- 個人情報について：カード番号や氏名などは含まれておらず、特徴量（V1〜V28）は主成分分析（PCA）により匿名化済みです

Kaggle公式ページから直接ダウンロードするにはKaggleアカウントでのログイン（またはAPIキーの設定）が必要なため、
本ノートブックでは、上記のライセンスに基づき、同じデータをzip形式で自社リポジトリ（GitHub）に保管し、
そちらから読み込んでいます。

## 使うデータ：クレジットカード取引データ（不正利用検知）

2013年9月にヨーロッパのカード会員が行った取引データです。実際の取引をもとに作成されており、
**約28万4千件**という、これまで扱ってきたデータ（数百〜数千件）とは桁違いのボリュームがあります。

| 列名 | 意味 |
| --- | --- |
| Time | 最初の取引からの経過秒数 |
| V1〜V28 | 取引の特徴量（プライバシー保護のため主成分分析で匿名化済み） |
| Amount | 取引金額 |
| **Class** | **不正利用かどうか（予測対象）　1=不正、0=正常** |

V1〜V28が匿名化されているのは、実際のクレジットカード情報や個人情報を直接扱えないためです。
企業の実データを扱う際には、このような**匿名化・マスキング処理**が必要になる場面が多くあります。

このデータもURLから直接読み込めるため、ダウンロード作業は不要です。

### コラム：主成分分析（PCA）とは

V1〜V28は「主成分分析（PCA: Principal Component Analysis）」という手法で作られた数値です。聞き慣れない言葉かもしれませんが、
考え方自体はシンプルです。

企業の取引データには、取引金額・時間帯・利用場所・加盟店の業種・会員の過去の利用傾向など、**数十個の項目（列）**が
あるのが普通です。項目が多すぎると、人間には全体像が把握しにくく、モデルの学習にも時間がかかります。

PCAは、こうした**多数の項目を、情報をなるべく失わないように少数の合成変数にまとめ直す**手法です。
たとえば「身長」と「体重」という2つの項目は、ある程度連動して動きます（背が高い人は体重も重い傾向がある）。
このような**相関のある項目同士をひとまとめにし、「体格の大きさ」のような1つの合成変数に置き換える**——
これがPCAのおおまかなイメージです。

今回のデータでは、この仕組みを利用して、

- 数十個あった元の項目を、V1〜V28という28個の合成変数に集約
- 集約の過程で元の項目の組み合わせ方が数式の中に溶け込むため、**逆算して元の項目を復元することが事実上できなくなる**

という2つの効果を同時に得ています。前者は「大量の項目を扱いやすくする」という分析上のメリット、
後者は「個人情報や企業の機密情報を保護する」という匿名化上のメリットです。この2つを同時に満たせる点が、
PCAが匿名化の手段としてもよく使われる理由です。

**ここで押さえておきたいポイント**

- PCAは「情報を圧縮する」手法であり、「消す」わけではない（合成変数の中に元の情報が数式として溶け込んでいる）
- 一方で、どの項目がどう組み合わさってV1になったのかは公開されない限り分からないため、結果として匿名化になる
- そのため、V1〜V28が具体的に何を意味するかは今回のデータからは読み取れない（前述の通り、Kaggle公式でも非公開）

この後の分析で「V14が重要な特徴量」といった結果が出てきますが、それが業務的に何を意味するかまでは
分からない、という点を念頭に置いて読み進めてください。

In [ ]:
# 必要なライブラリを読み込みます
# XGBoostとLightGBMはColabに標準搭載されていますが、念のためバージョンを確認しておきます
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

print('準備完了')

## ステップ①：データの読み込み

284,807件・31列というサイズのデータです（zip圧縮で約66MB、展開後は約144MB）。Excelで開こうとすると起動だけでも時間がかかり、
関数を1つ入れるだけでフリーズしてしまうこともある規模ですが、Colab上のPythonであれば圧縮されたzipファイルのまま数秒〜十数秒で読み込めます。

In [ ]:
url = 'https://raw.githubusercontent.com/nagomi-tech/blog-aibeginner/main/kaggle/archive.zip'

start = time.time()
data = pd.read_csv(url, compression='zip')
elapsed = time.time() - start

print(f'読み込み時間: {elapsed:.1f}秒')
print(f'データ件数: {len(data):,}件, 列数: {len(data.columns)}列')
data.head()

読み込みが完了すると、284,807件・31列のデータが表示されます。数十万件規模のデータでも、
Pythonであれば一瞬で読み込めることが実感できるかと思います。

## ステップ②：データの確認

学習を始める前に、欠損値の有無と、不正利用（Class=1）の割合を確認します。

In [ ]:
# 欠損値の確認
print('欠損値の合計:', data.isnull().sum().sum())

# 不正利用/正常の内訳
counts = data['Class'].value_counts()
fraud_rate = data['Class'].mean() * 100

print('\nClassの内訳:')
print(counts)
print(f'\n不正利用: {counts[1]:,}件 ({fraud_rate:.3f}%)')
print(f'正常取引: {counts[0]:,}件 ({100 - fraud_rate:.3f}%)')

実行すると、欠損値は0件、そして**不正利用はわずか0.17%程度**しかないことが確認できます。
28万件のうち不正利用はたった492件です。

これは典型的な**不均衡データ（Imbalanced Data）**です。実務のデータ分析では、こうした「レアケースの検出」が
目的になることが多く、その典型的なパターンです。

このような場合、「全部『正常』と予測しておけば正解率99.8%」という無意味なモデルができてしまうため、
**正解率（Accuracy）ではなく、AUCや適合率・再現率で評価する**ことが重要になります。

In [ ]:
# 不正利用/正常の内訳をグラフで確認
plt.figure(figsize=(5, 4))
counts.plot(kind='bar', color=['steelblue', 'salmon'])
plt.xticks([0, 1], ['正常 (0)', '不正利用 (1)'], rotation=0)
plt.ylabel('件数')
plt.title('取引件数の内訳（不正利用は非常に少ない）')
plt.tight_layout()
plt.show()

## ステップ③：学習データと検証データに分割

不均衡データを分割する際は、`stratify`オプションを指定し、学習データと検証データで
不正利用の割合が偏らないようにします。

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=['Class'])
y = data['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # 不正利用の割合を学習/検証データで揃える
)

print(f'学習データ: {len(X_train):,}件（不正利用 {y_train.sum()}件）')
print(f'検証データ: {len(X_test):,}件（不正利用 {y_test.sum()}件）')

## ステップ④：3つのモデルを学習する

ここからは**ランダムフォレスト・XGBoost・LightGBM**の3モデルを個別に学習し、精度と学習時間を比較します。
1つずつ`fit()`することで、それぞれのモデルの挙動の違いを体感してみましょう。

In [ ]:
from sklearn.ensemble import RandomForestClassifier

start = time.time()
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1  # PCの複数コアを使って並列に学習
)
rf_model.fit(X_train, y_train)
rf_time = time.time() - start

print(f'ランダムフォレストの学習時間: {rf_time:.1f}秒')

In [ ]:
import xgboost as xgb

start = time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
xgb_time = time.time() - start

print(f'XGBoostの学習時間: {xgb_time:.1f}秒')

In [ ]:
import lightgbm as lgb

start = time.time()
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    is_unbalance=True,  # 不正利用(Class=1)が極端に少ないデータであることをモデルに伝える
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train, y_train)
lgb_time = time.time() - start

print(f'LightGBMの学習時間: {lgb_time:.1f}秒')

3つとも同じ28万件のデータで学習していますが、学習時間にはっきりと差が出るはずです。
特にLightGBMは「Light（軽量）」の名の通り、大量データでの学習速度に強みがあるモデルです。
データ量がさらに増えるビジネスの現場では、この差が数分・数時間単位の差になることもあります。

## ステップ⑤：精度を比較する

不均衡データなので、正解率ではなく**AUC（ROC曲線の面積）**で比較します。

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

models = {
    'RandomForest': rf_model,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
}
times = {'RandomForest': rf_time, 'XGBoost': xgb_time, 'LightGBM': lgb_time}

results = []
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test, proba)
    results.append({'model': name, 'AUC': round(auc_score, 4), '学習時間(秒)': round(times[name], 1)})

result_df = pd.DataFrame(results).sort_values('AUC', ascending=False)
print(result_df.to_string(index=False))

実行すると、3モデルのAUCと学習時間が一覧表示されます（数値は実行のたびに多少変わります）。
一般的にこのデータでは3モデルともAUC 0.9前後の高い精度になりますが、学習時間には大きな差が見られるはずです。

「精度はある程度近くても、学習時間は大きく違う」——これは大量データを扱う実務において非常に重要な観点です。
精度だけでなく、**再学習にかかる時間やコストも含めてモデルを選ぶ**必要があります。

なお、LightGBMには`is_unbalance=True`というオプションを指定しています。今回のように不正利用が0.2%しかない
極端な不均衡データでは、何も指定しないとモデルが「多数派（正常）ばかりを学習してしまう」ことがあり、
このオプションで少数派（不正利用）をきちんと学習するよう調整しています。

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(7, 6))
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random (AUC=0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC曲線の比較')
plt.legend()
plt.tight_layout()
plt.show()

## ステップ⑥：適合率・再現率を確認する

不正利用検知の実務では、「不正を見逃さないこと（再現率）」と「正常な取引を誤って不正と判定しないこと（適合率）」の
バランスが重要になります。ここでは代表としてLightGBMの結果を詳しく見てみます。

In [ ]:
y_pred = lgb_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['正常', '不正利用']))

`recall`（再現率）が低い場合、「本当は不正なのに見逃してしまった取引」が多いことを意味します。
逆に`precision`（適合率）が低いと、「正常なのに不正と誤判定してしまった取引」が多く、
カードの利用者に無用な確認連絡が増えてしまいます。

どちらを重視するかは、ビジネス上の判断です。不正による損失額と、誤判定によって発生する
カスタマーサポートのコストを天秤にかけて、しきい値を調整する必要があります。

## ステップ⑦：特徴量重要度を比較する

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

importances = {
    'RandomForest': rf_model.feature_importances_,
    'XGBoost': xgb_model.feature_importances_,
    'LightGBM': lgb_model.feature_importances_,
}

for ax, (name, importance) in zip(axes, importances.items()):
    imp_series = pd.Series(importance, index=X.columns).sort_values(ascending=False).head(10)
    imp_series.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(name)
    ax.set_xlabel('重要度')

plt.tight_layout()
plt.show()

V1〜V28は主成分分析によって匿名化された特徴量のため、「V14が重要」と分かっても
それが具体的に何を意味するのかは、このデータからは読み取れません。

実務で自社データを扱う場合は、匿名化されていない生の特徴量（利用場所・時間帯・金額など）を使うため、
重要度の結果がそのままビジネス上のヒントになります。この点も、公開データセットと実データの違いとして
覚えておくとよいでしょう。

## ステップ⑧：新しい取引データで予測してみる

In [ ]:
# 検証データから3件抜き出して、実際にどう予測されるか確認します
sample = X_test.sample(3, random_state=1)

for name, model in models.items():
    pred = model.predict(sample)
    proba = model.predict_proba(sample)[:, 1]
    print(f'--- {name} ---')
    for i, (p, prob) in enumerate(zip(pred, proba)):
        label = '不正利用の疑いあり' if p == 1 else '正常'
        print(f'  取引{i+1}: {label}（不正確率 {prob:.3f}）')
    print()

## まとめ

今回は28万件という、これまでより大きなデータを使い、ランダムフォレスト・XGBoost・LightGBMの
3モデルを個別に学習・比較しました。

- **数十万件規模のデータでも、Pythonなら数秒〜数十秒で読み込み・処理できる**（Excelでは現実的でない規模）
- 実務データは**不均衡（レアケースの検出）**であることが多く、正解率だけで判断すると誤った結論を導く
- 精度が同程度でも、**学習時間・処理速度はモデルによって大きく異なる**——大量データほどこの差が実務に効いてくる
- 匿名化された特徴量では重要度の「意味」までは読み取れない。自社の生データであれば、ここがそのままビジネスの気づきにつながる

**⚠️ 注意：このノートブックは学習目的のサンプルです。実際の不正検知システムを構築する際は、
セキュリティ・プライバシー・法令遵守の観点から専門家を交えた検討が必要です。**

次回以降は、さらに実務に近いデータの前処理（欠損値・外れ値の扱いなど）や、
自社データをGoogle Driveから読み込んで分析する方法についても触れていく予定です。